In [1]:
## Run this cell FIRST

import subprocess
import time
import sys


print("Starting Safe YARN Cluster Cleanup...")

# 1. Clean up local zombie spark references before querying YARN
if 'spark' in sys.modules or 'pyspark' in sys.modules:
    try:
        from pyspark.sql import SparkSession
        active_session = SparkSession.getActiveSession()
        if active_session:
            print("Stopping local legacy session safely...")
            active_session.stop()
    except Exception:
        pass

try:
    # 2. Query YARN for active sessions
    cmd_list = "yarn application -list -appStates RUNNING,ACCEPTED"
    result = subprocess.check_output(cmd_list, shell=True, text=True)
    
    lines = result.strip().split("\n")
    apps_killed = 0
    
    for line in lines:
        if "application_" in line:
            app_id = line.split()[0]
            print(f"Killing conflicting YARN session: {app_id}")
            subprocess.call(f"yarn application -kill {app_id}", shell=True)
            apps_killed += 1
            
    if apps_killed > 0:
        print("Waiting for resource managers to clear memory allocations...")
        time.sleep(6)
        print("Cluster environment successfully normalized!")
    else:
        print("Cluster queue is already wide open and clean.")

except Exception as e:
    print(f"YARN query bypassed: {e}")


Starting Safe YARN Cluster Cleanup...


2026-08-24 01:16:33,762 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.20.0.4:8032


Killing conflicting YARN session: application_1787528572206_0005


2026-08-24 01:16:35,031 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.20.0.4:8032


Killing application application_1787528572206_0005


2026-08-24 01:16:35,615 INFO impl.YarnClientImpl: Killed application application_1787528572206_0005


Waiting for resource managers to clear memory allocations...
Cluster environment successfully normalized!


In [2]:
## Run this cell SECOND

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic

print("Allocating new Spark Session...")

spark = SparkSession.builder \
    .appName("Sedona-Fresh-Start") \
    .config("spark.master", "yarn") \
    .config("spark.submit.deployMode", "client") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark Session successfully attached to YARN!")
    

Allocating new Spark Session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/24 01:16:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/24 01:16:49 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive} is set, falling back to uploading libraries under SPARK_HOME.


Spark Session successfully attached to YARN!


In [3]:
## Run this cell THIRD

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic


@register_cell_magic
def sql(line, cell):

    # Eliminar comentarios SQL de líneas completas
    lines = []
    for line in cell.splitlines():
        if not line.strip().startswith("#"):
            lines.append(line)

    cell = "\n".join(lines)

    # Separar sentencias
    statements = [
        s.strip()
        for s in cell.split(";")
        if s.strip()
    ]

    for statement in statements:
        print("\n==============================")
        print("RUNNING:")
        print(statement)
        print("==============================")

        result = spark.sql(statement)
        # Only attempt to call show() if the query actually returns data rows
        if result.schema:
            result.show(truncate=False)
        else:
            print("Command executed successfully.")

In [4]:
## Run this cell FOURTH

from py4j.java_gateway import java_import

java_import(spark._jvm, "org.apache.hadoop.fs.FileSystem")
java_import(spark._jvm, "org.apache.hadoop.fs.Path")

fs = spark._jvm.FileSystem.get(
    spark.sparkContext._jsc.hadoopConfiguration()
)

fs.delete(
    spark._jvm.Path("/spark-warehouse"),
    True
)

fs.mkdirs(
    spark._jvm.Path("/spark-warehouse")
)

print("spark-warehouse clean")


spark-warehouse clean


In [ ]:
%%sql

# create trips in delta

CREATE OR REPLACE TEMP VIEW citibike_csv
USING csv
OPTIONS (
  path 'file:///opt/data/2023-citibike-tripdata',
  header 'true',
  inferSchema 'true'
);

DROP TABLE IF EXISTS trips;
CREATE TABLE trips
USING DELTA
AS 
SELECT * 
FROM citibike_csv
WHERE start_lng IS NOT NULL AND start_lat IS NOT NULL AND end_lng IS NOT NULL AND end_lat IS NOT NULL ;

ALTER TABLE trips ADD COLUMN origin GEOMETRY;
ALTER TABLE trips ADD COLUMN target GEOMETRY;

# For spark 3
# ALTER TABLE trips ADD COLUMN duration_ms INTEGER;

# For Spark 4, avoid Overflow
ALTER TABLE trips ADD COLUMN duration_ms BIGINT;


UPDATE trips SET 
  origin = st_setsrid(st_point(start_lng, start_lat), 4326),
  target = st_setsrid(st_point(end_lng, end_lat), 4326),
  duration_ms= (unix_timestamp(ended_at) - unix_timestamp(started_at)) * 1000,
  start_station_id = CASE WHEN start_station_id RLIKE '^[0-9]+[.][0-9]$'
    THEN concat(start_station_id, '0') ELSE start_station_id END,
  end_station_id = CASE WHEN end_station_id RLIKE '^[0-9]+[.][0-9]$'
    THEN concat(end_station_id, '0') ELSE end_station_id END;




In [6]:
%%sql

# o copiar en HDFS
# hdfs dfs -mkdir -p /opt/data
# hdfs dfs -put /opt/data/nynta2020_26b /opt/data/nynta2020_26b

CREATE OR REPLACE TEMP VIEW auxi_nta_2020 USING shapefile
OPTIONS (path 'file:///opt/data/nynta2020_26b', header 'true', inferSchema 'true');

DROP TABLE IF EXISTS nta;
CREATE TABLE nta
USING DELTA
AS SELECT * FROM auxi_nta_2020;


#Analyzing shp file we found: EPSG:102718 
SELECT DISTINCT st_srid(geometry) FROM nta;

# Nevertheless, the SRID automatically detected ESRI:102718  is not undertand by Sedona/GeooTools
# Set the EPSG:2263 srid before the transformation, which is equivalet to EPSG:2263. 
# Take into consideration that ST_Transform, in Sedona, requires the prefix EPSG

UPDATE nta SET geometry = ST_Transform(ST_SetSRID(geometry, 2263), 'EPSG:4326');


SELECT DISTINCT st_srid(geometry ) 
FROM nta;


RUNNING:
CREATE OR REPLACE TEMP VIEW auxi_nta_2020 USING shapefile
OPTIONS (path 'file:///opt/data/nynta2020_26b', header 'true', inferSchema 'true')
Command executed successfully.

RUNNING:
DROP TABLE IF EXISTS nta
Command executed successfully.

RUNNING:
CREATE TABLE nta
USING DELTA
AS SELECT * FROM auxi_nta_2020


Command executed successfully.

RUNNING:
SELECT DISTINCT st_srid(geometry) FROM nta


+-----------------+
|st_srid(geometry)|
+-----------------+
|102718           |
+-----------------+


RUNNING:
UPDATE nta SET geometry = ST_Transform(ST_SetSRID(geometry, 2263), 'EPSG:4326')


26/08/24 01:28:24 WARN UpdateCommand: Could not validate number of records due to missing statistics.


+-----------------+
|num_affected_rows|
+-----------------+
|262              |
+-----------------+


RUNNING:
SELECT DISTINCT st_srid(geometry ) 
FROM nta


+-----------------+
|st_srid(geometry)|
+-----------------+
|4326             |
+-----------------+



In [ ]:
%%sql

DROP TABLE IF EXISTS stations;

CREATE TABLE stations (
  id STRING,
  name STRING,
  latitude DOUBLE,
  longitude DOUBLE,
  geometry GEOMETRY,
  boroname STRING,
  ntacode STRING,
  ntaname STRING
)
USING DELTA;


## Un solo escaneo de trips, consolidado y persistido
CREATE OR REPLACE TEMP VIEW stations_clean AS
WITH trip_starts AS (
  SELECT
    TRIM(CAST(start_station_id AS STRING)) AS id,
    start_station_name AS name,
    start_lat AS latitude,
    start_lng AS longitude
  FROM trips
  WHERE start_station_id IS NOT NULL

  UNION ALL

  SELECT
    TRIM(CAST(end_station_id AS STRING)) AS id,
    end_station_name AS name,
    end_lat AS latitude,
    end_lng AS longitude
  FROM trips
  WHERE end_station_id IS NOT NULL
)
SELECT
  id,
  FIRST(name, true) AS name,
  AVG(latitude) AS latitude,
  AVG(longitude) AS longitude,
  COUNT(*) AS no_trips
FROM trip_starts
GROUP BY id;

# Identifiers of the same station: same name, less than 1 m apart
DROP TABLE IF EXISTS station_duplicates;
CREATE TABLE station_duplicates
USING DELTA
AS SELECT d.id AS dup_id, k.id AS keep_id
FROM stations_clean d JOIN stations_clean k
  ON d.name = k.name AND d.id <> k.id AND k.no_trips > d.no_trips AND
  ST_DistanceSphere(ST_Point(d.longitude, d.latitude),
    ST_Point(k.longitude, k.latitude)) < 1;

MERGE INTO trips t
USING station_duplicates d
ON t.start_station_id = d.dup_id
WHEN MATCHED THEN UPDATE SET t.start_station_id = d.keep_id;

MERGE INTO trips t
USING station_duplicates d
ON t.end_station_id = d.dup_id
WHEN MATCHED THEN UPDATE SET t.end_station_id = d.keep_id;


## MERGE contra la vista ya calculada (liviana, sin recalcular)
MERGE INTO stations s
USING stations_clean n
ON s.id = n.id
WHEN MATCHED THEN UPDATE SET
  s.name = n.name,
  s.latitude = n.latitude,
  s.longitude = n.longitude,
  s.geometry = ST_Point(n.longitude, n.latitude)
WHEN NOT MATCHED THEN INSERT (
  id, name, latitude, longitude, geometry, boroname, ntacode, ntaname
)
VALUES (
  n.id, n.name, n.latitude, n.longitude,
  ST_Point(n.longitude, n.latitude),
  NULL, NULL, NULL
);

In [8]:
%%sql

MERGE INTO stations s
USING nta n
ON ST_Within(s.geometry, n.geometry)
WHEN MATCHED THEN
  UPDATE SET 
    s.boroname = n.boroname,
    s.ntacode = n.nta2020,
    s.ntaname = n.ntaname;


RUNNING:
MERGE INTO stations s
USING nta n
ON ST_Within(s.geometry, n.geometry)
WHEN MATCHED THEN
  UPDATE SET 
    s.boroname = n.boroname,
    s.ntacode = n.nta2020,
    s.ntaname = n.ntaname


+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|2273             |2273            |0               |0                |
+-----------------+----------------+----------------+-----------------+



In [ ]:
%%sql

# The boroughs, merging the NTAs of each borough

DROP TABLE IF EXISTS boroughs;
CREATE TABLE boroughs
USING DELTA
AS SELECT boroname, ST_Union_Aggr(geometry) AS geometry
FROM nta
GROUP BY boroname;

SELECT boroname, geometry FROM boroughs ORDER BY boroname;


In [9]:
%%sql

describe stations;

describe trips;


RUNNING:
describe stations
+---------+---------+-------+
|col_name |data_type|comment|
+---------+---------+-------+
|id       |string   |NULL   |
|name     |string   |NULL   |
|latitude |double   |NULL   |
|longitude|double   |NULL   |
|geometry |geometry |NULL   |
|boroname |string   |NULL   |
|ntacode  |string   |NULL   |
|ntaname  |string   |NULL   |
+---------+---------+-------+


RUNNING:
describe trips
+------------------+---------+-------+
|col_name          |data_type|comment|
+------------------+---------+-------+
|ride_id           |string   |NULL   |
|rideable_type     |string   |NULL   |
|started_at        |timestamp|NULL   |
|ended_at          |timestamp|NULL   |
|start_station_name|string   |NULL   |
|start_station_id  |string   |NULL   |
|end_station_name  |string   |NULL   |
|end_station_id    |string   |NULL   |
|start_lat         |double   |NULL   |
|start_lng         |double   |NULL   |
|end_lat           |double   |NULL   |
|end_lng           |double   |NULL   |
|